# Pancreas CT viewer

Use this notebook after training and checkpoint selection. Pick a training example or a CT file in Drive, run the model, then scroll through the axial slices. Green outlines the pancreas label. Pink outlines the tumour/mass label.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive/Major")
os.environ["PANCREAS_PROJECT_ROOT"] = str(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements.txt"), "ipywidgets"])

## Choose a scan and predict

Leave `SCAN_FILE` empty to use a training example. To view your own NIfTI file, enter its full Drive path. The default uses the CPU so the saved time can be used for the demo notes.

In [ ]:
import json
import config
from app import demo_cases, run_demo

SCAN_FILE = ""
DEVICE = "cpu"
scan = Path(SCAN_FILE) if SCAN_FILE else next(iter(demo_cases().values()))
demo_folder = config.RESULTS_DIR / "demo"
output_mask = demo_folder / "segmentation.nii.gz"
ct, labels, spacing, report = run_demo(scan, output_mask, DEVICE)
report["scan_file"] = scan.name
(demo_folder / "run.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
from app import draw_slice

def show_slice(z, pancreas, tumour):
    fig = draw_slice(ct, labels, spacing, z, pancreas, tumour)
    display(fig)
    plt.close(fig)

widgets.interact(
    show_slice,
    z=widgets.IntSlider(min=0, max=ct.shape[2] - 1, value=ct.shape[2] // 2,
                        description="Slice", continuous_update=False),
    pancreas=widgets.Checkbox(value=True, description="Pancreas"),
    tumour=widgets.Checkbox(value=True, description="Tumour/mass"),
)

In [ ]:
from google.colab import files

DOWNLOAD_MASK = False
if DOWNLOAD_MASK:
    files.download(str(output_mask))
else:
    print("Saved in Drive:", output_mask)